In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!nvidia-smi

Wed Apr 22 16:03:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"Model loaded. Max seq length: {max_seq_length}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded. Max seq length: 2048


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA adapter configured.")

Unsloth 2026.4.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapter configured.


In [ ]:
from google.colab import files
uploaded = files.upload()

dataset_file = list(uploaded.keys())[0]
print(f"Uploaded file: {dataset_file}")

Saving qwen-dataset.jsonl to qwen-dataset (1).jsonl
Uploaded file: qwen-dataset (1).jsonl


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=dataset_file, split="train")
print(f"Tổng số mẫu: {len(dataset)}")
print("\nMẫu đầu tiên:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Tổng số mẫu: 832

Mẫu đầu tiên:
{'messages': [{'role': 'system', 'content': 'Bạn là chuyên gia pháp luật giao thông Việt Nam, trả lời chính xác và có căn cứ pháp lý.'}, {'role': 'user', 'content': 'Luật Trật tự, an toàn giao thông đường bộ điều chỉnh những vấn đề gì?'}, {'role': 'assistant', 'content': 'Luật Trật tự, an toàn giao thông đường bộ điều chỉnh các vấn đề bao gồm quy tắc giao thông, phương tiện và người tham gia giao thông đường bộ, công tác chỉ huy, điều khiển, tuần tra, kiểm soát giao thông, giải quyết tai nạn giao thông đường bộ, cũng như trách nhiệm quản lý nhà nước và trách nhiệm của các cơ quan, tổ chức, cá nhân có liên quan đến trật tự, an toàn giao thông đường bộ. Phạm vi điều chỉnh của Luật bao quát toàn diện các mối quan hệ pháp lý phát sinh trong lĩnh vực giao thông đường bộ, từ hành vi của người tham gia giao thông đến hoạt động quản lý nhà nước chuyên ngành.'}]}


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print("=== Mẫu sau khi apply chat template ===")
print(dataset[0]["text"][:1500])

Map:   0%|          | 0/832 [00:00<?, ? examples/s]

=== Mẫu sau khi apply chat template ===
<|im_start|>system
Bạn là chuyên gia pháp luật giao thông Việt Nam, trả lời chính xác và có căn cứ pháp lý.<|im_end|>
<|im_start|>user
Luật Trật tự, an toàn giao thông đường bộ điều chỉnh những vấn đề gì?<|im_end|>
<|im_start|>assistant
Luật Trật tự, an toàn giao thông đường bộ điều chỉnh các vấn đề bao gồm quy tắc giao thông, phương tiện và người tham gia giao thông đường bộ, công tác chỉ huy, điều khiển, tuần tra, kiểm soát giao thông, giải quyết tai nạn giao thông đường bộ, cũng như trách nhiệm quản lý nhà nước và trách nhiệm của các cơ quan, tổ chức, cá nhân có liên quan đến trật tự, an toàn giao thông đường bộ. Phạm vi điều chỉnh của Luật bao quát toàn diện các mối quan hệ pháp lý phát sinh trong lĩnh vực giao thông đường bộ, từ hành vi của người tham gia giao thông đến hoạt động quản lý nhà nước chuyên ngành.<|im_end|>



In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "epoch",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/832 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=6):   0%|          | 0/832 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/832 [00:00<?, ? examples/s]

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | Total VRAM: {max_memory} GB")
print(f"VRAM đã dùng trước training: {start_gpu_memory} GB")

GPU: Tesla T4 | Total VRAM: 14.563 GB
VRAM đã dùng trước training: 5.393 GB


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 832 | Num Epochs = 3 | Total steps = 312
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,1.485789
10,1.291130
15,1.278298
20,1.206315
25,1.142510
30,1.162341
35,1.122358
40,1.140359
45,1.131041
50,1.030958


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"Thời gian train: {trainer_stats.metrics['train_runtime']:.2f} giây ({trainer_stats.metrics['train_runtime']/60:.2f} phút)")
print(f"VRAM peak: {used_memory} GB")
print(f"VRAM dùng cho LoRA: {used_memory_for_lora} GB")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Thời gian train: 1889.09 giây (31.48 phút)
VRAM peak: 9.064 GB
VRAM dùng cho LoRA: 3.671 GB
Final loss: 0.7885


In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Bạn là chuyên gia pháp luật giao thông Việt Nam, trả lời chính xác và có căn cứ pháp lý."},
    {"role": "user", "content": "Doanh nghiệp kinh doanh vận tải hành khách mà không có phù hiệu xe thì bị xử phạt như thế nào?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.7,
    top_p = 0.9,
    repetition_penalty = 1.05,
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Theo quy định tại điểm a khoản 5 Điều 30 Nghị định xử phạt vi phạm hành chính trong lĩnh vực giao thông đường bộ, doanh nghiệp, hợp tác xã, hộ kinh doanh kinh doanh vận tải hành khách bằng xe ô tô mà không gắn ký hiệu, biển số của chủ thể kinh doanh vận tải (thường gọi là phù hiệu) theo quy định hoặc gắn phù hiệu khác với phù hiệu thực tế của xe bị phạt tiền từ 1.000.000 đồng đến 2.000.000 đồng. Ngoài phạt tiền, theo điểm a khoản 7 Điều 30, người vi phạm còn bị tước Giấy phép lái xe từ 1 đến 3 tháng. Đây là biện pháp trừng phạt rất nghiêm khắc vì phù hiệu là công cụ quản lý nhà nước bắt buộc để tiện theo dõi, kiểm soát hoạt động vận tải, đảm bảo chất lượng dịch vụ và phòng chống tai nạn, vi phạm trên đường.<|im_end|>


In [ ]:
model.save_pretrained("lora_qwen25_vn_legal")
tokenizer.save_pretrained("lora_qwen25_vn_legal")

!zip -r lora_qwen25_vn_legal.zip lora_qwen25_vn_legal

from google.colab import files
files.download("lora_qwen25_vn_legal.zip")

  adding: lora_qwen25_vn_legal/ (stored 0%)
  adding: lora_qwen25_vn_legal/README.md (deflated 65%)
  adding: lora_qwen25_vn_legal/tokenizer.json (deflated 81%)
  adding: lora_qwen25_vn_legal/chat_template.jinja (deflated 71%)
  adding: lora_qwen25_vn_legal/tokenizer_config.json (deflated 43%)
  adding: lora_qwen25_vn_legal/adapter_model.safetensors (deflated 7%)
  adding: lora_qwen25_vn_legal/adapter_config.json (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Sau khi có file đã được train, sử dụng đoạn code sau để test

In [1]:
from unsloth import FastLanguageModel

# Load lại model đã finetune
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_qwen25_vn_legal",  # Thư mục LoRA adapter
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# Hàm tiện ích để hỏi
def ask(question, system_prompt="Bạn là chuyên gia pháp luật giao thông Việt Nam, trả lời chính xác và có căn cứ pháp lý."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        use_cache=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

# Dùng
print(ask("Người từ 16 đến dưới 18 tuổi lái xe mô tô có dung tích xi lanh từ 50 cm3 trở lên bị xử phạt thế nào"))

ModuleNotFoundError: No module named 'unsloth'

**Đánh giá mô hình bằng Perplexity**

In [3]:
import math
import torch
from datasets import load_dataset

# Tách 10-20% dataset làm test set (NÊN làm trước khi train)
# Nếu chưa tách, lấy 10 mẫu cuối làm test cho demo
test_dataset = load_dataset("json", data_files="qwen-dataset.jsonl", split="train")
test_dataset = test_dataset.select(range(len(test_dataset) - 20, len(test_dataset)))

def compute_perplexity(model, tokenizer, dataset):
    model.eval()
    total_loss = 0
    total_tokens = 0

    for item in dataset:
        messages = item["messages"]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss.item()
            num_tokens = inputs["input_ids"].size(1)
            total_loss += loss * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity, avg_loss

# Perplexity càng THẤP → model càng hiểu domain
ft_ppl, ft_loss = compute_perplexity(ft_model, ft_tokenizer, test_dataset)
print(f"Finetuned - Perplexity: {ft_ppl:.2f}, Loss: {ft_loss:.4f}")

# Load lại base model và đo
del ft_model, ft_tokenizer
free_memory()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048, load_in_4bit=True,
)
from unsloth.chat_templates import get_chat_template
base_tokenizer = get_chat_template(base_tokenizer, chat_template="qwen-2.5")

base_ppl, base_loss = compute_perplexity(base_model, base_tokenizer, test_dataset)
print(f"Base - Perplexity: {base_ppl:.2f}, Loss: {base_loss:.4f}")

print(f"\n Cải thiện perplexity: {((base_ppl - ft_ppl) / base_ppl * 100):.1f}%")

Generating train split: 0 examples [00:00, ? examples/s]

Failed to load JSON from file '/content/qwen-dataset.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: The document is empty.
ERROR:datasets.packaged_modules.json.json:Failed to load JSON from file '/content/qwen-dataset.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: The document is empty.


DatasetGenerationError: An error occurred while generating the dataset

**Hoặc so sánh theo theo cảm quan: nhìn câu trả lời bằng mắt**

In [ ]:
from unsloth import FastLanguageModel
import torch
import gc

# =========================================
# HÀM TIỆN ÍCH
# =========================================
def generate_answer(model, tokenizer, question, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            do_sample=True,
            use_cache=True,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()

SYSTEM_PROMPT = "Bạn là chuyên gia pháp luật giao thông Việt Nam, trả lời chính xác và có căn cứ pháp lý."

TEST_QUESTIONS = [
    "Công ty vận tải không có Giấy phép kinh doanh vận tải thì bị xử phạt thế nào?",
    "Lập bến xe lậu (bến dù, bến cóc) bị xử phạt thế nào?",
    "Vượt đèn đỏ xe máy bị phạt bao nhiêu tiền?",
    "Không đội mũ bảo hiểm khi đi xe máy bị phạt bao nhiêu?",
    "Doanh nghiệp vận tải hành khách không có phù hiệu xe bị xử phạt thế nào?",
]

In [ ]:
# =========================================
# BƯỚC 1: Lấy câu trả lời từ BASE MODEL
# =========================================
print("=" * 80)
print("LOADING BASE MODEL...")
print("=" * 80)

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

from unsloth.chat_templates import get_chat_template
base_tokenizer = get_chat_template(base_tokenizer, chat_template="qwen-2.5")

base_answers = []
for i, q in enumerate(TEST_QUESTIONS):
    print(f"\n[Base] Q{i+1}: {q}")
    ans = generate_answer(base_model, base_tokenizer, q, SYSTEM_PROMPT)
    base_answers.append(ans)
    print(f"→ {ans[:200]}...")

# Giải phóng VRAM
del base_model, base_tokenizer
free_memory()

In [ ]:
# =========================================
# BƯỚC 2: Lấy câu trả lời từ MODEL ĐÃ FINETUNE
# =========================================
print("=" * 80)
print("LOADING FINETUNED MODEL...")
print("=" * 80)

ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name="lora_qwen25_vn_legal",  # Thư mục LoRA của bạn
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

ft_answers = []
for i, q in enumerate(TEST_QUESTIONS):
    print(f"\n[Finetuned] Q{i+1}: {q}")
    ans = generate_answer(ft_model, ft_tokenizer, q, SYSTEM_PROMPT)
    ft_answers.append(ans)
    print(f"→ {ans[:200]}...")

In [ ]:
# =========================================
# BƯỚC 3: In ra đối chiếu song song
# =========================================
for i, q in enumerate(TEST_QUESTIONS):
    print("=" * 80)
    print(f"CÂU HỎI {i+1}: {q}")
    print("=" * 80)
    print("\n📘 BASE MODEL:")
    print(base_answers[i])
    print("\n🎓 FINETUNED MODEL:")
    print(ft_answers[i])
    print()